# 04 Evaluation Notebook

This notebook compares the recommender approaches in the project using the same train/validation/test artifacts:

- Popularity baseline
- Matrix Factorization
- Upgraded Hybrid recommender

It is designed to work with the packaged `src/recommender.py` module and the processed files in `data/processed/`.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd

BASE_DIR = os.path.abspath("..")
SRC_DIR = os.path.join(BASE_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

from recommender import HybridRecommender, hit_rate_at_k

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Imports ready.")
print("BASE_DIR:", BASE_DIR)
print("SRC_DIR :", SRC_DIR)


## Load processed artifacts

In [ ]:
PROCESSED = os.path.join(BASE_DIR, "data", "processed")

train = pd.read_csv(os.path.join(PROCESSED, "train.csv"))
val = pd.read_csv(os.path.join(PROCESSED, "val.csv"))
test = pd.read_csv(os.path.join(PROCESSED, "test.csv"))
movies_clean = pd.read_csv(os.path.join(PROCESSED, "movies_clean.csv"))
genre_encoded = pd.read_csv(os.path.join(PROCESSED, "genre_encoded.csv"))

for df, col in [(train, "rated_at"), (val, "rated_at"), (test, "rated_at")]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

print("Artifacts loaded successfully.")
print("train:", train.shape)
print("val  :", val.shape)
print("test :", test.shape)
print("movies_clean :", movies_clean.shape)
print("genre_encoded:", genre_encoded.shape)


## Train the upgraded recommender

In [ ]:
model = HybridRecommender(
    train_df=train,
    movies_df=movies_clean,
    genre_df=genre_encoded,
    n_components=50,
    neighbor_k=15,
    min_similarity=0.10,
    min_rating_for_profile=4.0,
    cf_weight=0.30,
    mf_weight=0.45,
    genre_weight=0.15,
    popularity_weight=0.10,
).fit()

print("Model trained successfully.")
print("Users in train:", len(model.user_ids))
print("Movies in train matrix:", len(model.movie_ids))


## Quick recommendation checks

In [ ]:
sample_user_id = int(train["userId"].iloc[0])

print(f"Sample user: {sample_user_id}")

print("\nPopularity-style fallback excluded seen items:")
display(model._popularity_fallback(top_n=10, exclude_ids=set(train.loc[train["userId"] == sample_user_id, "movieId"])))

print("\nMatrix Factorization recommendations:")
display(model.recommend_mf(sample_user_id, top_n=10))

print("\nHybrid recommendations:")
display(model.recommend_hybrid(sample_user_id, top_n=10))


## Optional: lightweight item-item KNN baseline

Your original notebook built an item-item nearest-neighbor recommender directly inside the notebook.  
This cell recreates that baseline so you can compare **Popularity vs KNN vs MF vs Hybrid** in one place.


In [ ]:
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

user_item_train = train.pivot_table(index="userId", columns="movieId", values="rating")
user_item_train_filled = user_item_train.fillna(0)

item_user_matrix = user_item_train_filled.T
item_user_sparse = csr_matrix(item_user_matrix.to_numpy())

knn_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=20,
    n_jobs=-1
)
knn_model.fit(item_user_sparse)

user_ids = user_item_train.index.to_list()
movie_ids = user_item_train.columns.to_list()

movie_to_idx = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}
idx_to_movie = {idx: movie_id for movie_id, idx in movie_to_idx.items()}

popularity_df = model.popularity_df.copy()

def get_similar_movies_knn(movie_id, n_neighbors=10):
    if movie_id not in movie_to_idx:
        return pd.DataFrame(columns=["movieId", "similarity"])

    movie_idx = movie_to_idx[movie_id]
    distances, indices = knn_model.kneighbors(item_user_sparse[movie_idx], n_neighbors=n_neighbors + 1)

    rows = []
    for dist, idx in zip(distances.flatten(), indices.flatten()):
        neighbor_movie_id = idx_to_movie[idx]
        if neighbor_movie_id == movie_id:
            continue
        rows.append((neighbor_movie_id, 1 - dist))

    return pd.DataFrame(rows, columns=["movieId", "similarity"]).sort_values("similarity", ascending=False).reset_index(drop=True)

def recommend_knn(user_id, top_n=10, neighbor_k=10, min_similarity=0.10):
    user_history = train[train["userId"] == user_id].copy()

    if user_history.empty:
        return popularity_df.head(top_n)[["movieId", "title", "genres", "weighted_score"]]

    rated_movie_ids = set(user_history["movieId"].tolist())
    candidate_scores = {}

    for _, row in user_history.iterrows():
        seed_movie_id = row["movieId"]
        user_rating = row["rating"]
        similar_movies = get_similar_movies_knn(seed_movie_id, n_neighbors=neighbor_k)

        for _, sim_row in similar_movies.iterrows():
            candidate_movie_id = sim_row["movieId"]
            similarity = sim_row["similarity"]

            if candidate_movie_id in rated_movie_ids:
                continue
            if similarity < min_similarity:
                continue

            weighted_contribution = similarity * user_rating

            if candidate_movie_id not in candidate_scores:
                candidate_scores[candidate_movie_id] = {"weighted_sum": 0.0, "similarity_sum": 0.0}

            candidate_scores[candidate_movie_id]["weighted_sum"] += weighted_contribution
            candidate_scores[candidate_movie_id]["similarity_sum"] += similarity

    if not candidate_scores:
        fallback = popularity_df[~popularity_df["movieId"].isin(rated_movie_ids)]
        return fallback.head(top_n)[["movieId", "title", "genres", "weighted_score"]]

    predictions = []
    for movie_id, values in candidate_scores.items():
        predicted_score = values["weighted_sum"] / values["similarity_sum"]
        predictions.append((movie_id, predicted_score))

    recs = pd.DataFrame(predictions, columns=["movieId", "knn_score"])
    recs = recs.merge(movies_clean[["movieId", "title", "genres"]], on="movieId", how="left")
    recs = recs.merge(popularity_df[["movieId", "weighted_score", "rating_count"]], on="movieId", how="left")
    recs = recs.sort_values(["knn_score", "weighted_score"], ascending=False).head(top_n).reset_index(drop=True)
    return recs

def hit_rate_knn(eval_df, top_n=10, min_eval_rating=4.0, sample_users=100):
    eligible_users = []
    for user_id, user_eval in eval_df.groupby("userId"):
        if (user_eval["rating"] >= min_eval_rating).sum() == 0:
            continue
        if user_id in set(train["userId"].unique()):
            eligible_users.append(user_id)

    eligible_users = eligible_users[:sample_users]
    hits, total = 0, 0

    for user_id in eligible_users:
        actual_positive_movies = set(
            eval_df[(eval_df["userId"] == user_id) & (eval_df["rating"] >= min_eval_rating)]["movieId"].tolist()
        )
        if not actual_positive_movies:
            continue

        recs = recommend_knn(user_id=user_id, top_n=top_n)
        recommended_movies = set(recs["movieId"].tolist())

        hit = len(actual_positive_movies.intersection(recommended_movies)) > 0
        hits += int(hit)
        total += 1

    return hits / total if total > 0 else 0.0

print("KNN baseline ready.")


## Compare models on validation and test

Metrics here use **Hit Rate@10**, matching the project style from your earlier notebook.


In [ ]:
sample_users = 100
top_n = 10
min_eval_rating = 4.0

results = pd.DataFrame([
    {
        "model": "Popularity",
        "validation_hit_rate@10": hit_rate_at_k(model, val, top_n=top_n, min_eval_rating=min_eval_rating, sample_users=sample_users, method="popularity"),
        "test_hit_rate@10": hit_rate_at_k(model, test, top_n=top_n, min_eval_rating=min_eval_rating, sample_users=sample_users, method="popularity"),
    },
    {
        "model": "KNN Item-Item",
        "validation_hit_rate@10": hit_rate_knn(val, top_n=top_n, min_eval_rating=min_eval_rating, sample_users=sample_users),
        "test_hit_rate@10": hit_rate_knn(test, top_n=top_n, min_eval_rating=min_eval_rating, sample_users=sample_users),
    },
    {
        "model": "Matrix Factorization",
        "validation_hit_rate@10": hit_rate_at_k(model, val, top_n=top_n, min_eval_rating=min_eval_rating, sample_users=sample_users, method="mf"),
        "test_hit_rate@10": hit_rate_at_k(model, test, top_n=top_n, min_eval_rating=min_eval_rating, sample_users=sample_users, method="mf"),
    },
    {
        "model": "Upgraded Hybrid",
        "validation_hit_rate@10": hit_rate_at_k(model, val, top_n=top_n, min_eval_rating=min_eval_rating, sample_users=sample_users, method="hybrid"),
        "test_hit_rate@10": hit_rate_at_k(model, test, top_n=top_n, min_eval_rating=min_eval_rating, sample_users=sample_users, method="hybrid"),
    },
])

results = results.sort_values("validation_hit_rate@10", ascending=False).reset_index(drop=True)
display(results)


## Add stronger ranking metrics

Hit Rate@10 is useful, but interviewers often like to see more ranking-oriented evaluation too.

This section adds:

- **Precision@K**: how many recommended movies were actually relevant
- **Recall@K**: how many relevant movies were recovered
- **Catalog Coverage@K**: how much of the movie catalog the recommender exposes across users


In [ ]:
def evaluate_ranking_metrics(model_obj, eval_df, top_n=10, min_eval_rating=4.0, sample_users=100, method="hybrid"):
    eligible_users = []
    train_user_set = set(model_obj.train_df["userId"].unique())

    for user_id, user_eval in eval_df.groupby("userId"):
        relevant = set(user_eval.loc[user_eval["rating"] >= min_eval_rating, "movieId"].tolist())
        if not relevant:
            continue
        if user_id not in train_user_set:
            continue
        eligible_users.append(user_id)

    eligible_users = eligible_users[:sample_users]

    precision_scores = []
    recall_scores = []
    all_recommended_items = set()

    for user_id in eligible_users:
        relevant_items = set(
            eval_df[
                (eval_df["userId"] == user_id) &
                (eval_df["rating"] >= min_eval_rating)
            ]["movieId"].tolist()
        )

        if not relevant_items:
            continue

        if method == "popularity":
            seen_items = set(model_obj.train_df.loc[model_obj.train_df["userId"] == user_id, "movieId"].tolist())
            recs = model_obj._popularity_fallback(top_n=top_n, exclude_ids=seen_items)
        elif method == "mf":
            recs = model_obj.recommend_mf(user_id=user_id, top_n=top_n)
        else:
            recs = model_obj.recommend_hybrid(user_id=user_id, top_n=top_n)

        recommended_items = set(recs["movieId"].tolist())
        all_recommended_items.update(recommended_items)

        hits = len(recommended_items.intersection(relevant_items))
        precision_scores.append(hits / top_n if top_n > 0 else 0.0)
        recall_scores.append(hits / len(relevant_items) if len(relevant_items) > 0 else 0.0)

    catalog_size = len(set(model_obj.movies_df["movieId"].tolist()))
    coverage = len(all_recommended_items) / catalog_size if catalog_size > 0 else 0.0

    return {
        "precision@k": float(np.mean(precision_scores)) if precision_scores else 0.0,
        "recall@k": float(np.mean(recall_scores)) if recall_scores else 0.0,
        "coverage@k": float(coverage),
        "users_evaluated": len(precision_scores),
    }


def evaluate_knn_metrics(eval_df, top_n=10, min_eval_rating=4.0, sample_users=100):
    eligible_users = []
    train_user_set = set(train["userId"].unique())

    for user_id, user_eval in eval_df.groupby("userId"):
        relevant = set(user_eval.loc[user_eval["rating"] >= min_eval_rating, "movieId"].tolist())
        if not relevant:
            continue
        if user_id not in train_user_set:
            continue
        eligible_users.append(user_id)

    eligible_users = eligible_users[:sample_users]

    precision_scores = []
    recall_scores = []
    all_recommended_items = set()

    for user_id in eligible_users:
        relevant_items = set(
            eval_df[
                (eval_df["userId"] == user_id) &
                (eval_df["rating"] >= min_eval_rating)
            ]["movieId"].tolist()
        )

        if not relevant_items:
            continue

        recs = recommend_knn(user_id=user_id, top_n=top_n)
        recommended_items = set(recs["movieId"].tolist())
        all_recommended_items.update(recommended_items)

        hits = len(recommended_items.intersection(relevant_items))
        precision_scores.append(hits / top_n if top_n > 0 else 0.0)
        recall_scores.append(hits / len(relevant_items) if len(relevant_items) > 0 else 0.0)

    catalog_size = len(set(movies_clean["movieId"].tolist()))
    coverage = len(all_recommended_items) / catalog_size if catalog_size > 0 else 0.0

    return {
        "precision@k": float(np.mean(precision_scores)) if precision_scores else 0.0,
        "recall@k": float(np.mean(recall_scores)) if recall_scores else 0.0,
        "coverage@k": float(coverage),
        "users_evaluated": len(precision_scores),
    }

print("Advanced metric helpers ready.")


## Compare advanced metrics on validation and test

In [ ]:
advanced_results = []

for split_name, split_df in [("Validation", val), ("Test", test)]:
    pop_metrics = evaluate_ranking_metrics(
        model_obj=model,
        eval_df=split_df,
        top_n=10,
        min_eval_rating=4.0,
        sample_users=100,
        method="popularity"
    )
    knn_metrics = evaluate_knn_metrics(
        eval_df=split_df,
        top_n=10,
        min_eval_rating=4.0,
        sample_users=100
    )
    mf_metrics = evaluate_ranking_metrics(
        model_obj=model,
        eval_df=split_df,
        top_n=10,
        min_eval_rating=4.0,
        sample_users=100,
        method="mf"
    )
    hybrid_metrics = evaluate_ranking_metrics(
        model_obj=model,
        eval_df=split_df,
        top_n=10,
        min_eval_rating=4.0,
        sample_users=100,
        method="hybrid"
    )

    advanced_results.extend([
        {"split": split_name, "model": "Popularity", **pop_metrics},
        {"split": split_name, "model": "KNN Item-Item", **knn_metrics},
        {"split": split_name, "model": "Matrix Factorization", **mf_metrics},
        {"split": split_name, "model": "Upgraded Hybrid", **hybrid_metrics},
    ])

advanced_results_df = pd.DataFrame(advanced_results)
display(advanced_results_df)


## Presentation takeaway

Use these metrics together when you explain your project:

- **Hit Rate@10** shows whether the model can place at least one relevant movie in the top recommendations
- **Precision@10** shows recommendation quality
- **Recall@10** shows how much of the user’s relevant set you recover
- **Coverage@10** shows whether the system keeps recommending the same narrow set of movies or exposes more of the catalog

A strong interview summary is:

> “I compared recommendation models not just on hit rate, but also on precision, recall, and coverage so I could measure both ranking quality and catalog diversity.”


## Save trained model artifact

Run this if you want the app to load a pre-trained recommender from disk.


In [ ]:
MODELS_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

model_path = os.path.join(MODELS_DIR, "hybrid_recommender.pkl")
model.save(model_path)

print("Saved model to:")
print(model_path)
